# Fourier Encoding in PyTorch

In [ ]:
import math

import numpy as np
import torch
from PIL import Image
from torch import Tensor, nn

from models.deep_learning.architectures import MLP
from models.deep_learning.components.embeddings.fourier_encoding import FourierPositionalEncoding
from models.deep_learning.components.embeddings.visualization import (
    plot_embedding,
    plot_embeddingdims,
)

device = torch.device("mps")

In [ ]:
def transformer_frequency(embed_dim: int, ndim: int = 1, base: int = 10000) -> Tensor:
    """COmpute frequencies for Transformer-style Fourier Positional Encoding
    embed_dim: total embedding dimension (must be divisible by 2*ndim)
    ndim: number of spatial dimensions (e.g. 1 for sequences, 2 for images)
    base: base for frequency scaling (default 10000)"""
    assert embed_dim % (2 * ndim) == 0, "Embedding dimension must be divisible by 2*ndim"
    half_dim = embed_dim // (2 * ndim)
    freq = torch.exp(-math.log(base) * torch.arange(half_dim) / half_dim)
    return freq

In [ ]:
fourier_posenc = FourierPositionalEncoding(freq=transformer_frequency(24, 1)).to(device)
emb = fourier_posenc(spatial_dimensions=(50,))
emb.shape

## Embedding Visualization (per dimension)

In [ ]:
position = FourierPositionalEncoding.make_coords((50,), torch.device("cpu"))

plot_embeddingdims(position, emb.detach().cpu(), title="Fourier Positional Embedding")

## Embedding Visualization (heat map)

In the 1D case (sequence), this formulation is equivalent to the sinusoidal positional encoding used in the Transformer, as it employs the same frequency scaling.

In [ ]:
fourier_posenc = FourierPositionalEncoding(freq=transformer_frequency(130)).to(device)
emb = fourier_posenc(spatial_dimensions=(50,))

plot_embedding(emb.detach().cpu(), "Fourier(sinusoidal) Positional Embedding")

In [ ]:
SM = emb @ emb.T

In [ ]:
plot_embedding(
    SM.detach().cpu(),
    "Similarity between Positional Encodings",
    cmap="Blues",
    vmin=SM.min(),
    vmax=SM.max(),
)

## Test

Lets learn a figure map (from spatial position to pixel colors)

In [ ]:
im = Image.open("rose_crop.jpeg").convert("RGB")
im_small = im.resize((128, 128))
im_small

## Learn without positional encoding

In [ ]:
rosenet = MLP(input_dim=2, output_dim=3, hidden_dims=[256, 128, 64], activation_cls=nn.ReLU)

In [ ]:
rose_tensor = (torch.as_tensor(np.array(im_small), dtype=torch.float32) / 255.0 - 0.5).to(device)
xs = torch.linspace(-1, 1, im_small.size[1])
ys = torch.linspace(-1, 1, im_small.size[0])

norm_position = torch.stack(
    torch.meshgrid(ys, xs, indexing="ij"),
    dim=-1,
).to(device)
norm_position.shape

## Training

In [ ]:
net = rosenet.to(device)
optim = torch.optim.Adam(net.parameters(), lr=1e-3)

for it in range(5000):
    optim.zero_grad()
    loss = abs(net(norm_position) - rose_tensor).mean()
    loss.backward()
    optim.step()
    if it % 100 == 0:
        print(f"Iteration {it}, loss: {loss.item():.6f}")

In [ ]:
Image.fromarray(
    ((net(norm_position) + 0.5).clamp(0, 1).cpu().detach().numpy() * 255).astype(np.uint8)
)

In [ ]:
im_small

## Training with Sinusoidal positional encoding

In [ ]:
emb_dim = 16
fourier_posenc = FourierPositionalEncoding(freq=transformer_frequency(emb_dim, ndim=2)).to(device)
rosenet = MLP(
    input_dim=emb_dim, output_dim=3, hidden_dims=[256, 128, 64], activation_cls=nn.ReLU
).to(device)
fourier_posenc(spatial_dimensions=im_small.size).shape

In [ ]:
rose_tensor = rose_tensor.to(device)
optim = torch.optim.Adam(rosenet.parameters(), lr=1e-3)

for it in range(5000):
    optim.zero_grad()
    loss = abs(rosenet(fourier_posenc(im_small.size)) - rose_tensor).mean()
    loss.backward()
    optim.step()
    if it % 100 == 0:
        print(f"Iteration {it}, loss: {loss.item():.6f}")

In [ ]:
img_gen = rosenet(fourier_posenc(im_small.size)).detach().cpu()
img_gen.shape

In [ ]:
Image.fromarray(((img_gen + 0.5).clamp(0, 1).cpu().detach().numpy() * 255).astype(np.uint8))

In [ ]:
im_small

## Generalization (bigger image)

In [ ]:
img_gen = rosenet(fourier_posenc(im_small.size, sampling_factor=8))
img_gen.shape

In [ ]:
Image.fromarray(((img_gen + 0.5).clamp(0, 1).cpu().detach().numpy() * 255).astype(np.uint8))

In [ ]:
im.resize((1024, 1024))

In [ ]:
enb_flat = fourier_posenc(im_small.size).flatten(end_dim=1)
enb_flat.shape

In [ ]:
im_small.size

## Flatten similarity

After flattening the spatial dimensions, long-range correlations become more visible: sinusoidal encodings capture local positional differences well, but their periodic nature introduces global ambiguities, causing distant positions to appear similar.

In [ ]:
SM = enb_flat @ enb_flat.T
sm = SM[::2, ::2]
plot_embedding(
    sm.detach().cpu(),
    "Similarity between Positional Encodings",
    cmap="Blues",
    vmin=sm.min(),
    vmax=sm.max(),
)